# 04. LoCoMotif Motif Discovery

Run real LoCoMotif motif discovery in regime-agnostic and regime-conditioned modes. This notebook records failures instead of producing fake or proxy LoCoMotif outputs.


## Execution Mode

Local mode is a smoke test. HPC mode uses the full configured asset/frequency/parameter sweep.


In [ ]:
EXECUTION_MODE = "auto"  # allowed: "auto", "local", "hpc"
STAGE_NAME = "04_locomotif_motif_discovery"
from pathlib import Path
import sys, warnings
warnings.simplefilter("default")

def find_workflow_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "hpc_config.py").exists(): return candidate
    fallback = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis\HPC workflow\HPC_Regime_and_motif_discovery")
    if (fallback / "src" / "hpc_config.py").exists(): return fallback
    raise RuntimeError("Could not locate workflow root.")

WORKFLOW_ROOT = find_workflow_root(); SRC_DIR = WORKFLOW_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))
from hpc_config import build_workflow_paths, detect_execution_mode, environment_info, find_project_root, load_workflow_config, output_suffix, save_config_snapshot
from io_utils import ensure_workflow_dirs, setup_stage_logger
PROJECT_ROOT = find_project_root(WORKFLOW_ROOT)
MODE = detect_execution_mode(EXECUTION_MODE, PROJECT_ROOT)
CONFIG = load_workflow_config(MODE, WORKFLOW_ROOT)
PATHS = build_workflow_paths(WORKFLOW_ROOT, PROJECT_ROOT)
ensure_workflow_dirs(PATHS)
SUFFIX = output_suffix(CONFIG)
LOGGER = setup_stage_logger(PATHS.logs, STAGE_NAME, SUFFIX)
SNAPSHOT_PATH = save_config_snapshot(CONFIG, PATHS, STAGE_NAME, MODE)
env = environment_info(PATHS)
print("Stage:", STAGE_NAME)
print("Execution mode:", MODE)
print("Project root:", PATHS.project_root)
print("Workflow root:", PATHS.workflow_root)
print("Input feature root:", PATHS.data_root)
print("Config snapshot:", SNAPSHOT_PATH)
print({"python": env["python_version"].split()[0], "platform": env["platform"], "hostname": env["hostname"], "cpu_count": env["cpu_count"], "memory_available_gb": env["memory_available_gb"], "gpu": env["gpu"]})


## Discover Feature Files


In [ ]:
from data_loading import discover_feature_files
from io_utils import save_table
inventory = discover_feature_files(PATHS.project_root, CONFIG)
if inventory.empty:
    raise FileNotFoundError(f"No feature parquet files discovered under {PATHS.data_root}")
inventory_for_save = inventory.copy(); inventory_for_save["path"] = inventory_for_save["path"].astype(str)
save_table(inventory_for_save, PATHS.tables / f"input_feature_inventory{SUFFIX}.csv", PATHS.tables / f"input_feature_inventory{SUFFIX}.parquet")
print("Discovered input files:")
print(inventory_for_save[["asset", "frequency", "market", "path"]].to_string(index=False))


## Run LoCoMotif and Final Comparisons

The notebook checks for `locomotif.locomotif.apply_locomotif`, runs only the real API, and writes final thesis comparison tables when Matrix Profile outputs exist.


In [ ]:
import pandas as pd
from data_loading import apply_mode_limits, describe_feature_frame, load_feature_file
from feature_selection import ensure_core_features, select_motif_feature_matrix
from io_utils import failure_record, save_failure_log, save_json, save_table
from locomotif_utils import INSTALL_HINT, locomotif_api_status, run_locomotif_discovery
from motif_evaluation import compare_mp_locomotif, evaluate_locomotif_results, evaluate_matrix_profile_results, thesis_key_results_table
from plotting_utils import plot_locomotif_intervals
from regime_utils import continuous_segments, default_regime_methods, load_regime_labels, merge_regime_labels

import_timeout_seconds = int(CONFIG.get("locomotif", {}).get("import_timeout_seconds", 20 if CONFIG.get("active_mode") == "local" else 120))
api_status = locomotif_api_status(timeout_seconds=import_timeout_seconds)
LOCOMOTIF_READY = bool(api_status.get("available"))
print("LoCoMotif API status:", api_status)
save_json(api_status, PATHS.logs / f"04_locomotif_api_status{SUFFIX}.json")
quantile_labels = load_regime_labels(WORKFLOW_ROOT, "quantile", SUFFIX)
hmm_labels = load_regime_labels(WORKFLOW_ROOT, "hmm", SUFFIX)
print(f"Loaded Quantile labels: {len(quantile_labels):,}")
print(f"Loaded HMM labels: {len(hmm_labels):,}")

all_results, all_status, all_feature_diagnostics = [], [], []
dataset_summaries, failures = [], []
figure_count = 0
max_figures = int(CONFIG.get("plotting", {}).get("max_figures_per_notebook", 40))
loco_cfg = CONFIG.get("locomotif", {})
max_points = loco_cfg.get("local_max_points") if CONFIG.get("active_mode") == "local" else loco_cfg.get("hpc_max_points")

def run_loco_slice(slice_df, asset, frequency, mode, regime_method, regime_label, segment_id):
    global figure_count
    local_results = []
    if not LOCOMOTIF_READY:
        error_message = api_status.get("error") or "Real LoCoMotif API is unavailable."
        failures.append(failure_record(STAGE_NAME, RuntimeError(error_message), asset, frequency, {"mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "install_hint": INSTALL_HINT, "api_status": api_status}))
        all_status.append({"asset": asset, "frequency": frequency, "mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "status": "failed", "error": error_message, "install_hint": INSTALL_HINT})
        return []
    try:
        prepared = ensure_core_features(slice_df, rolling_window=int(CONFIG.get("quantile", {}).get("default_rolling_window", 60))).reset_index(drop=True)
        timestamps = prepared["timestamp"].reset_index(drop=True)
        X, scaled, feature_columns, feature_diag = select_motif_feature_matrix(prepared, CONFIG)
        feature_diag.insert(0, "segment_id", segment_id); feature_diag.insert(0, "regime_label", regime_label); feature_diag.insert(0, "regime_method", regime_method); feature_diag.insert(0, "frequency", frequency); feature_diag.insert(0, "asset", asset)
        all_feature_diagnostics.append(feature_diag)
    except Exception as exc:
        failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "step": "feature_selection"}))
        return []

    for params in loco_cfg.get("parameter_grid", []):
        try:
            context = {"asset": asset, "frequency": frequency, "mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "feature_set": ",".join(feature_columns)}
            result, status = run_locomotif_discovery(X, timestamps, params, context, max_points=max_points)
            status.update(context); status.update(params); status["status"] = "success"; all_status.append(status)
            local_results.append(result)
            if figure_count < max_figures and not result.empty:
                value_col = "close" if "close" in prepared.columns else "log_return"
                prefix = f"04_{asset}_{frequency}_{mode}_{regime_method}_{regime_label}_l{params.get('l_min')}_{params.get('l_max')}{SUFFIX}"
                plot_locomotif_intervals(result, prepared[value_col], timestamps, f"{asset} {frequency} LoCoMotif {mode} {regime_label}", PATHS.figures / f"{prefix}_intervals.png"); figure_count += 1
        except Exception as exc:
            failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "params": params, "install_hint": INSTALL_HINT}))
            failed_status = {"asset": asset, "frequency": frequency, "mode": mode, "regime_method": regime_method, "regime_label": regime_label, "segment_id": segment_id, "status": "failed", "error": str(exc)}
            failed_status.update(params); all_status.append(failed_status)
    return [frame for frame in local_results if frame is not None and not frame.empty]

for meta in inventory.to_dict("records"):
    asset, frequency = meta["asset"], meta["frequency"]
    try:
        raw = load_feature_file(meta["path"])
        df = apply_mode_limits(raw, CONFIG)
        if df.empty: raise ValueError("No rows remain after execution-mode limits.")
        dataset_summaries.append(describe_feature_frame(meta, df))
        print(f"\n{asset} {frequency}: rows={len(df):,}")
        all_results.extend(run_loco_slice(df, asset, frequency, "agnostic", "none", "all", "full_series"))
        for source, labels in [("quantile", quantile_labels), ("hmm", hmm_labels)]:
            if labels.empty: continue
            for regime_method in default_regime_methods(labels, CONFIG, source):
                merged = merge_regime_labels(df, labels, asset, frequency, regime_method)
                if "regime_label" not in merged.columns or merged["regime_label"].isna().all(): continue
                # Regime-conditioned LoCoMotif uses continuous same-regime segments only; disconnected periods are never concatenated.
                segments = continuous_segments(merged["regime_label"], int(loco_cfg.get("min_segment_length", 160)), merged["timestamp"])
                max_segments = loco_cfg.get("local_max_conditioned_segments_per_group") if CONFIG.get("active_mode") == "local" else None
                if max_segments: segments = segments.groupby("regime_label", group_keys=False).head(int(max_segments))
                for segment in segments.to_dict("records"):
                    segment_df = merged.iloc[int(segment["start_pos"]): int(segment["end_pos_exclusive"])].reset_index(drop=True)
                    all_results.extend(run_loco_slice(segment_df, asset, frequency, "conditioned", regime_method, segment["regime_label"], segment["segment_id"]))
    except Exception as exc:
        LOGGER.exception("LoCoMotif batch failure for %s %s", asset, frequency)
        failures.append(failure_record(STAGE_NAME, exc, asset, frequency, {"path": str(meta["path"]), "install_hint": INSTALL_HINT}))

LOCO_RESULT_COLUMNS = ["asset", "frequency", "method", "mode", "regime_method", "regime_label", "segment_id", "feature_set", "motif_set_rank", "motif_instance_id", "role", "motif_start", "motif_end", "motif_length", "motif_start_timestamp", "motif_end_timestamp", "motif_score", "motif_set_size", "l_min", "l_max", "rho", "nb", "overlap", "warping", "runtime_seconds", "n_observations", "status"]
LOCO_EVAL_COLUMNS = ["asset", "frequency", "method", "mode", "regime_method", "regime_label", "l_min", "l_max", "rho", "number_of_motifs", "mean_motif_distance_or_score", "recurrence_count", "mean_motif_length", "median_motif_length", "runtime_seconds", "time_split_stability", "cross_regime_overlap", "notes"]
results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame(columns=LOCO_RESULT_COLUMNS)
status_table = pd.DataFrame(all_status)
feature_diagnostics = pd.concat(all_feature_diagnostics, ignore_index=True) if all_feature_diagnostics else pd.DataFrame(columns=["asset", "frequency", "regime_method", "regime_label", "segment_id", "feature", "missing_fraction", "unique_values", "kept", "center", "scale", "scaler"])
evaluation = evaluate_locomotif_results(results)
if evaluation.empty:
    evaluation = pd.DataFrame(columns=LOCO_EVAL_COLUMNS)
runtime = status_table.copy() if not status_table.empty else pd.DataFrame(columns=["asset", "frequency", "mode", "regime_method", "regime_label", "segment_id", "status", "error"])

save_table(results, PATHS.motifs_locomotif / f"locomotif_motif_results{SUFFIX}.csv", PATHS.motifs_locomotif / f"locomotif_motif_results{SUFFIX}.parquet")
save_table(evaluation, PATHS.motifs_locomotif / f"locomotif_evaluation{SUFFIX}.csv", PATHS.motifs_locomotif / f"locomotif_evaluation{SUFFIX}.parquet")
save_table(runtime, PATHS.motifs_locomotif / f"locomotif_runtime{SUFFIX}.csv", PATHS.motifs_locomotif / f"locomotif_runtime{SUFFIX}.parquet")
save_table(feature_diagnostics, PATHS.tables / f"04_locomotif_feature_diagnostics{SUFFIX}.csv", PATHS.tables / f"04_locomotif_feature_diagnostics{SUFFIX}.parquet")
save_table(pd.DataFrame(dataset_summaries), PATHS.tables / f"04_locomotif_dataset_summary{SUFFIX}.csv", PATHS.tables / f"04_locomotif_dataset_summary{SUFFIX}.parquet")
failure_df = save_failure_log(failures, PATHS.logs / f"04_locomotif_failures{SUFFIX}.csv")

mp_path = PATHS.motifs_matrix_profile / f"matrix_profile_motif_results{SUFFIX}.parquet"
mp_results = pd.read_parquet(mp_path) if mp_path.exists() else pd.DataFrame()
mp_eval = evaluate_matrix_profile_results(mp_results) if not mp_results.empty else pd.DataFrame()
mp_vs_loco = compare_mp_locomotif(mp_results, results) if not mp_results.empty and not results.empty else pd.DataFrame()
key_table = thesis_key_results_table(mp_eval, evaluation)

agnostic_vs_conditioned = pd.DataFrame(); regime_method_summary = pd.DataFrame()
if not key_table.empty:
    agnostic_vs_conditioned = key_table.groupby(["method", "mode"], dropna=False).agg(number_of_motifs=("number_of_motifs", "sum"), runtime_seconds=("runtime_seconds", "sum"), time_split_stability=("time_split_stability", "mean")).reset_index()
    regime_method_summary = key_table.groupby(["method", "regime_method", "mode"], dropna=False).agg(number_of_motifs=("number_of_motifs", "sum"), runtime_seconds=("runtime_seconds", "sum")).reset_index()

runtime_comparison_frames = []
mp_runtime_path = PATHS.motifs_matrix_profile / f"matrix_profile_runtime{SUFFIX}.parquet"
if mp_runtime_path.exists():
    mp_runtime = pd.read_parquet(mp_runtime_path); mp_runtime["method"] = "matrix_profile"; runtime_comparison_frames.append(mp_runtime)
if not runtime.empty:
    loco_runtime = runtime.copy(); loco_runtime["method"] = "locomotif"; runtime_comparison_frames.append(loco_runtime)
runtime_comparison = pd.concat(runtime_comparison_frames, ignore_index=True, sort=False) if runtime_comparison_frames else pd.DataFrame()

save_table(mp_vs_loco, PATHS.comparisons / f"mp_vs_locomotif_summary{SUFFIX}.csv", PATHS.comparisons / f"mp_vs_locomotif_summary{SUFFIX}.parquet")
save_table(agnostic_vs_conditioned, PATHS.comparisons / f"agnostic_vs_conditioned_summary{SUFFIX}.csv", PATHS.comparisons / f"agnostic_vs_conditioned_summary{SUFFIX}.parquet")
save_table(regime_method_summary, PATHS.comparisons / f"regime_method_comparison_summary{SUFFIX}.csv", PATHS.comparisons / f"regime_method_comparison_summary{SUFFIX}.parquet")
save_table(runtime_comparison, PATHS.comparisons / f"runtime_comparison{SUFFIX}.csv", PATHS.comparisons / f"runtime_comparison{SUFFIX}.parquet")
save_table(key_table, PATHS.comparisons / f"thesis_key_results_table{SUFFIX}.csv", PATHS.comparisons / f"thesis_key_results_table{SUFFIX}.parquet")

print("\nLoCoMotif result rows:", len(results))
print("LoCoMotif evaluation rows:", len(evaluation))
print("Failures:", len(failure_df))
if results.empty:
    print("No real LoCoMotif motif rows were produced. This usually means dtai-locomotif is unavailable or all real API calls failed.")
    print("Install hint:", INSTALL_HINT)
key_table.head(20)
